# Python进阶复习：面向对象、并发与生成器


本篇根据课程提供的《Python进阶_复习笔记》整理，作为前 1～7 天的复习入口。内容按原资料顺序覆盖面向对象、闭包与装饰器、网络编程、进程与线程、生成器与正则，以及数据结构与算法中的单向链表。代码单元全部使用 Markdown 围栏展示，不在 Notebook 中执行。


## 一、面向对象复习

面向过程强调按步骤完成任务，例如自己编写排序过程；面向对象则把数据和行为封装到对象中，通过对象协作完成任务。类是对象的模板，对象是类的具体实例。

定义类时，属性通常是名词，方法通常是动词；创建对象会自动调用 `__init__`，`self` 表示当前对象。


```python
class Student:
    def __init__(self, name, age):
        self.name = name
        self.age = age

    def study(self):
        print(f'{self.name} 正在学习...')

    def eat(self):
        print(f'{self.name} 正在吃饭...')


if __name__ == '__main__':
    s1 = Student('乔峰', 38)
    print(s1.name, s1.age)
    s1.study()
    s1.eat()
```

`Student` 的属性属于对象本身；调用 `s1.study()` 时，Python 会把 `s1` 作为第一个参数传给 `self`。`__init__` 是双下划线方法，名称不要误写成 `int`。

### 1. 继承

子类通过 `class 子类名(父类名):` 继承父类。父类中定义一次的属性和方法，子类对象可以直接使用，从而复用代码。子类没有自己的 `__init__` 时，会沿继承关系查找可用的初始化方法。


```python
class Person:
    def __init__(self):
        self.class_name = 'AI23期'

    def eat(self):
        print('人要吃饭')


class Student(Person):
    pass


if __name__ == '__main__':
    s1 = Student()
    print(s1.class_name)
    s1.eat()
```

继承关系表达“是一种”关系：`Student` 是 `Person` 的一种。课程示例中 `Student()` 可以直接使用父类的 `class_name` 和 `eat()`。

### 2. 私有化

名称以 `__` 开头的实例属性或方法会触发名称改写，外部不能用原名直接访问。课程用它来保护 ATM 的余额，并通过公共方法提供受控访问。严格来说，Python 的双下划线是名称改写机制，不是绝对的安全边界；学习阶段应遵循“通过公共接口访问”的约定。


```python
class ATM:
    def __init__(self):
        self.__money = 10000

    def get_money(self):
        return self.__money

    def set_money(self, money):
        self.__money = money


if __name__ == '__main__':
    xm = ATM()
    # print(xm.__money)  # 私有属性不能通过这个名字直接访问
    xm.set_money(100)
    print(xm.get_money())
```

`get_money()` 用于读取，`set_money()` 用于修改。真实业务中，修改方法还应加入金额范围、权限等校验；这里仅保留课程的私有化示例。

### 3. 多继承、重名成员与 `super()`

多继承写在类名后的括号中。查找成员时遵循方法解析顺序（MRO），通常先查靠前的父类，再继续向后查找。`super()` 适合按 MRO 调用下一个父类；需要明确指定某个父类时可以写 `父类名.方法(self)`。


```python
class Person:
    def __init__(self):
        self.class_name = 'AI23期'

    def eat(self):
        print('人要吃饭')


class School:
    def __init__(self):
        self.class_name = 'AI学校'

    def study(self):
        print('学习')

    def eat(self):
        print('学生为了好好学习, 不吃饭!')


class Student(School, Person):
    def show(self):
        # super().eat()      # 按 MRO 找到最近的父类方法
        # Person.eat(self)    # 精确调用 Person
        School.eat(self)      # 精确调用 School


if __name__ == '__main__':
    s1 = Student()
    print(s1.class_name)
    s1.eat()
    s1.study()
    s1.show()
```

示例中 `School` 写在 `Person` 前面，所以 `s1.eat()` 找到 `School.eat()`；`show()` 中则演示精确调用。多个父类都定义 `__init__` 时，应根据需要显式调用各父类初始化方法，避免只初始化其中一部分。

## 二、闭包和装饰器复习

### 1. 闭包

闭包由三点组成：函数嵌套、内部函数引用外部函数的局部变量、外部函数返回内部函数对象。外部函数调用结束后，内部函数仍然可以通过闭包保存并使用那个局部变量，因此延长了它的生命周期。


```python
def fn_outer():
    a = 100

    def fn_inner():
        print(a)

    return fn_inner


if __name__ == '__main__':
    my_fn = fn_outer()
    my_fn()
```

调用 `fn_outer()` 得到的是函数对象，不是 `fn_inner()` 的执行结果；随后调用 `my_fn()` 才会访问保存下来的 `a`。

### 2. 装饰器

装饰器是闭包的一种常见用法：在不修改目标函数主体的情况下，为它增加登录校验、日志或计时等功能。传统写法是 `comment = check_login(comment)`；在函数上写 `@check_login` 是同一过程的语法糖。


```python
def check_login(fn_name):
    def fn_inner():
        print('校验登陆')
        fn_name()

    return fn_inner


@check_login
def comment():
    print('发表评论')


if __name__ == '__main__':
    comment()
```

装饰器函数接收目标函数，返回一个新的函数。新函数先完成额外功能，再调用原函数。课程示例预期先输出登录校验，再发表评论；这里仅记录代码，不运行它。

### 3. 多个装饰器的顺序

```python
@check_login
@check_code
def comment():
    ...
```

等价于先执行 `comment = check_code(comment)`，再执行 `comment = check_login(comment)`；真正调用时从最外层开始，因此输出顺序是“校验登录 → 校验验证码 → 原函数”。记忆方式是：装饰阶段由内向外，调用阶段由外向内。


```python
def check_login(fn_name):
    def fn_inner():
        print('校验登陆')
        fn_name()

    return fn_inner


def check_code(fn_name):
    def fn_inner():
        print('校验验证码')
        fn_name()

    return fn_inner


@check_login
@check_code
def comment():
    print('发表评论')


if __name__ == '__main__':
    comment()
```

随堂资料用两个装饰器演示“先校验登录，后校验验证码”。每个包装函数都要返回内部函数，否则目标函数名会被替换成 `None`。

## 三、网络编程：TCP Socket

课程用“一句话交情”示例说明 TCP 客户端和服务器的一次通信。双方必须约定相同的 IP、端口和协议；字符串发送前编码为字节，接收后再解码。

服务器端的顺序是：创建 Socket → `bind` 绑定地址 → `listen` 监听 → `accept` 接受连接 → `recv` 接收 → `send` 回复 → 关闭连接。客户端的顺序是：创建 Socket → `connect` 连接 → `send` 发送 → `recv` 接收 → 关闭。


### 1. 客户端

```python
import socket

client_socket = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
client_socket.connect(('127.0.0.1', 12345))
client_socket.send('你好服务器端, 我是客户端!'.encode('utf-8'))
data = client_socket.recv(1024).decode('utf-8')
print(f'客户端收到: {data}')
client_socket.close()
```

客户端的 `recv(1024)` 表示本次最多读取 1024 个字节；一次 `recv` 不等于完整的应用层消息，真实协议通常还要设计长度或结束标记。课程示例只演示一次短消息。


### 2. 服务器端

```python
import socket

server_socket = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
server_socket.bind(('127.0.0.1', 12345))
server_socket.listen(5)
accept_socket, client_info = server_socket.accept()
data = accept_socket.recv(1024).decode('utf-8')
print(f'服务器端收到: {data}')
accept_socket.send('欢迎来到Socket的世界'.encode('utf-8'))
accept_socket.close()
server_socket.close()
```

`listen(5)` 中的数字是等待连接队列的容量示意；`accept()` 会阻塞等待客户端。`accept_socket` 专门负责和已连接客户端通信，`server_socket` 负责监听新连接。


## 四、进程和线程

进程和线程都是实现多任务的手段。并行是多个任务在多核上同时执行；并发是多个任务交替获得执行机会。课程用“多车道”和“单车道”帮助区分两者。

| 对比 | 进程 | 线程 |
| --- | --- | --- |
| 主要职责 | 资源分配的基本单位 | CPU 调度的基本单位 |
| 数据关系 | 不同进程默认相互隔离 | 同一进程内线程共享进程数据 |
| 资源与稳定性 | 通常开销更大、隔离更强 | 通常开销更小、共享更方便 |

进程和线程都可能需要同步；“共享”不等于可以忽略竞态条件。


### 1. 带参数的多进程

创建 `multiprocessing.Process` 时，`target` 指定目标函数，`args` 用元组传位置参数，`kwargs` 用字典传关键字参数，最后调用 `start()` 启动。

```python
import multiprocessing
import time


def code(name, num):
    for i in range(num):
        time.sleep(0.1)
        print(f'{name} 正在敲第{i}行代码...')


def music(name, count):
    for i in range(count):
        time.sleep(0.1)
        print(f'{name} 正在听第{i}首歌**********')


if __name__ == '__main__':
    p1 = multiprocessing.Process(target=code, args=('乔峰', 10))
    p2 = multiprocessing.Process(
        target=music,
        kwargs={'count': 10, 'name': '虚竹'},
    )
    p1.start()
    p2.start()
```

课程代码只调用 `start()`，没有 `join()`；如果主程序需要等待子进程结束并按顺序收尾，应在实际项目中根据需求调用 `join()`。Windows 创建多进程时要保留 `if __name__ == '__main__':` 保护。

### 2. 带参数的多线程

线程的创建方式与进程类似，把类替换为 `threading.Thread`。线程共享同一进程的内存，因此传递和修改共享对象更直接，也更需要考虑同步。

```python
import threading
import time


def code(name, num):
    for i in range(num):
        time.sleep(0.1)
        print(f'{name} 正在敲第{i}行代码...')


def music(name, count):
    for i in range(count):
        time.sleep(0.1)
        print(f'{name} 正在听第{i}首歌**********')


if __name__ == '__main__':
    t1 = threading.Thread(target=code, args=('乔峰', 10))
    t2 = threading.Thread(
        target=music,
        kwargs={'count': 10, 'name': '虚竹'},
    )
    t1.start()
    t2.start()
```

课程示例中两个任务各自循环并休眠，输出顺序由调度决定，不能把交错顺序当成固定结果。

## 五、生成器

生成器按照规则逐个产生数据，而不是一次性创建完整序列，适合数据量大或不能一次放入内存的场景。课程展示两种写法：生成器表达式和带 `yield` 的生成器函数。

- 列表推导式会立即生成所有元素；
- 生成器表达式只保存生成规则和当前状态；
- 函数执行到 `yield` 会暂停，下次迭代从暂停处继续；
- 一个生成器通常只能顺序消费一次。


```python
import sys

my_list = [i for i in range(0, 100000, 2)]
print(sys.getsizeof(my_list))

my_generator = (i for i in range(0, 100000, 2))
for i in my_generator:
    print(i)
print(sys.getsizeof(my_generator))


def get_generator():
    for i in range(0, 100000, 2):
        yield i


my_generator = get_generator()
```

随堂资料打印了整个偶数列表和生成器的内存大小，输出量很大；本篇不运行这些打印。`yield` 同时使函数变成生成器函数，调用 `get_generator()` 时得到生成器对象，循环时才逐个计算。

## 六、正则表达式

正则表达式（Regular Expression，简称 `re`）用于按规则校验、查找、匹配、过滤或替换字符串。课程基本用法是导入 `re`，用 `re.match()` 从字符串开头尝试匹配，匹配成功后通过 `group()` 读取结果。

手机号规则 `^1[3-9]\d{9}$` 的含义是：以 `1` 开头，第二位为 `3`～`9`，再跟 9 位数字，并且 `$` 要求匹配到字符串末尾。


```python
import re

phone = '13812345678'
result = re.match(r'^1[3-9]\d{9}$', phone)
if result:
    print(f'手机号正确, 匹配到: {result.group()}')
else:
    print('手机号错误')

s1 = 'qq:12345678'
result = re.match(r'(qq):(\d{6,11})', s1)
if result:
    print(result.group())
    print(result.group(0))
    print(result.group(1))
    print(result.group(2))
```

正则中的 `\d{6,11}` 表示前面的数字模式出现 6 到 11 次（包含两端），不是“至少 11 次”。`group(0)` 是完整匹配，`group(1)`、`group(2)` 分别读取两个括号分组。

## 七、数据结构与算法复习

数据结构是存储和组织数据的方式，是算法处理业务时的数据载体；算法是解决问题的思路和步骤。课程再次强调用大 `O` 记号关注随问题规模增长的主要项，常见复杂度从优到劣为：`O(1)`、`O(log n)`、`O(n)`、`O(n log n)`、`O(n²)`、`O(n³)`。

线性结构中的结点通常最多一个直接前驱和一个直接后继，例如顺序表、栈、队列和链表；树、图属于非线性结构。顺序表需要连续空间，链表的结点可以分散存放并通过链接关系连接。


## 八、单向链表回顾

单向链表由结点组成，每个结点包含数值域 `item` 和地址域 `next`；链表对象保存入口 `head`。空链表的 `head` 为 `None`。遍历时用游标 `cur` 沿 `next` 向后移动。


```python
class SingleNode:
    def __init__(self, item):
        self.item = item
        self.next = None


class SingleLinkedList:
    def __init__(self, node=None):
        self.head = node

    def is_empty(self):
        return self.head == None

    def length(self):
        count = 0
        cur = self.head
        while cur is not None:
            count += 1
            cur = cur.next
        return count


if __name__ == '__main__':
    node1 = SingleNode('乔峰')
    node2 = SingleNode('虚竹')
    my_linkedlist = SingleLinkedList(node1)
    print(f'数值域: {node1.item}')
    print(f'地址域: {node1.next}')
    print(my_linkedlist.is_empty())
    print(my_linkedlist.length())
```

`is_empty()` 只检查头指针；`length()` 从头到尾逐结点计数，时间复杂度为 `O(n)`，额外空间为 `O(1)`。课程后续的头插、尾插、指定位置插入、删除和查找实现已在第 06 篇笔记中按操作展开，本篇保留复习入口。

## 九、复习中的易错点

1. `__init__` 是初始化方法；创建对象时自动调用，`self` 不需要由调用者显式传入。
2. 继承解决复用，多继承要关注 MRO；需要指定父类时用 `父类名.方法(self)` 或恰当使用 `super()`。
3. 私有属性通过类提供的公共方法访问，不要把双下划线理解成绝对的安全机制。
4. 装饰器必须返回包装函数；多个装饰器的装饰顺序和调用顺序相反。
5. Socket 中字节与字符串要用 `encode` / `decode` 转换；`accept()` 和 `recv()` 可能阻塞。
6. 多进程和多线程的输出顺序由调度决定，不能伪造固定输出；Windows 多进程要保留主入口保护。
7. 生成器是惰性产生数据；正则分组用 `group(0)` 和 `group(1)` 等区分完整匹配与捕获组。
8. 链表访问必须从 `head` 沿 `next` 逐个移动，不能像顺序表一样按下标随机访问。
